In [1]:
import torch
from torch import nn
import math
from torch.nn import functional as F

自注意力机制

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V):
        d_k = Q.size(-1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        attn = self.softmax(scores)

        attn = self.dropout(attn)

        out = torch.matmul(attn, V)

        return out, attn

多头注意力机制

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.self_attn = SelfAttention(dropout)

        self.fc = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v):
        batch_size = q.size(0)

        Q = self.W_q(q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        out, attn = self.self_attn(Q, K, V)

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_k)

        out = self.fc(out)

        out = self.dropout(out)

        return out, attn

前馈神经网络

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))

PatchEmbedding

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, inchannels, d_model):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.n_patches = (img_size // patch_size) ** 2

        self.projection = nn.Conv2d(inchannels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.projection(x)

        x = x.flatten(2)

        return x.transpose(1, 2)

编码层

In [7]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff, dropout)

    def forward(self, src):
        out = src + self.attn(self.norm1(src), self.norm1(src), self.norm1(src))[0]
        out = out + self.ffn(self.norm2(out))

        return out

ViT模型

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, inchannels=3, d_model=768, n_heads=12, d_ff=3072, n_classes=1000, num_layers=12, dropout=0.1):
        super().__init__()
        self.patch_embedding = PatchEmbedding(img_size, patch_size, inchannels, d_model)
        n_patches = self.patch_embedding.n_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embedding = nn.Parameter(torch.zeros(1, 1+n_patches, d_model))

        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, n_classes)

        #  初始化
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embedding, std=0.02)

        self.apply(self._init_weight)

    def _init_weight(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        if isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        batch_size = x.size(0)

        x = self.patch_embedding(x)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)

        x = torch.cat([cls_tokens, x], dim=1)

        x = x + self.pos_embedding

        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        cls_out = x[:, 0, :]

        return self.head(cls_out)